In [1]:
import pandas as pd
import numpy as np
import glob
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Load dataset by glob

In [2]:
path = "/kaggle/input/io-t-sleep-stage-classification-version-2/train/train"
csv_files = glob.glob(f"{path}/*.csv")
df_train = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)
df_train

,BVP,ACC_X,ACC_Y,ACC_Z,TEMP,EDA,HR,IBI,Sleep_Stage
0,-16.787332,-29.654620,-35.583667,44.486126,30.773655,0.769030,52.986465,0.911322,W
1,2.375216,-30.549130,-35.593414,44.482602,30.773662,0.769137,52.986463,0.911322,W
2,46.691694,-29.650487,-35.582574,44.487709,30.773652,0.768899,52.986385,0.911322,W
3,126.648750,-29.604712,-35.591203,44.482427,30.773663,0.769318,52.986546,0.911322,W
4,208.143510,-30.267480,-35.588414,44.486243,30.773655,0.767400,52.986326,0.911322,W
...,...,...,...,...,...,...,...,...,...
31907035,47.832421,-26.690641,-36.058908,46.649187,34.629026,0.135512,55.329368,1.128359,W
31907036,60.657461,-26.691021,-35.617491,45.716012,34.629011,0.135450,55.329161,1.127777,W
31907037,50.879365,-26.691015,-36.191107,46.045236,34.629013,0.135492,55.329507,1.127192,W
31907038,38.674395,-26.690752,-36.093464,46.573528,34.629020,0.135473,55.329154,1.128328,W


# Select feature scaling to fit_transform

In [3]:
feature_cols = ["BVP", "ACC_X", "ACC_Y", "ACC_Z", "TEMP", "EDA", "HR", "IBI"]
scaler = StandardScaler()
df_train[feature_cols] = scaler.fit_transform(df_train[feature_cols])

# Select encode labels to fit_transform

In [4]:
label_encoder = LabelEncoder()
df_train["Sleep_Stage"] = label_encoder.fit_transform(df_train["Sleep_Stage"])

# Prepare windows sample data to train_test_split

In [6]:
X = df_train[feature_cols].values  #feature
y = df_train["Sleep_Stage"].values  #label
num_samples = len(X) // 480
X = X[: num_samples * 480].reshape(num_samples, 480, -1)  # Reshape for LSTM
y = y[::480]

# Train/Test split data by using 50% of data

In [9]:
# Use 50% of data
X, _, y, _ = train_test_split(X, y, test_size=0.5, random_state=42)

# Convert labels to categorical to train_test_split

In [11]:
y_categorical = tf.keras.utils.to_categorical(y, num_classes=3)
y_categorical

array([[0., 1., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [1., 0., 0.],
       [0., 1., 0.]])

# Split dataset

In [12]:
X_train, X_val, y_train, y_val = train_test_split(X, y_categorical, test_size=0.2, random_state=42)

# Define LSTM Model

In [14]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(480, X.shape[2])),
    BatchNormalization(),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [15]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [16]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 480, 64)             │          18,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 480, 64)             │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 480, 64)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 64)                  │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │           8,320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 3)                   │             387 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 60,675 (237.01 KB)

 Trainable params: 60,547 (236.51 KB)

 Non-trainable params: 128 (512.00 B)

# Train LSTM Model with 20 Epochs and batch size = 64

In [17]:
model.fit(X_train, y_train, epochs=20, batch_size=64, validation_data=(X_val, y_val), shuffle=True)

Epoch 1/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 16s 52ms/step - accuracy: 0.6169 - loss: 0.8912 - val_accuracy: 0.6622 - val_loss: 0.8195
Epoch 2/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - accuracy: 0.6642 - loss: 0.8144 - val_accuracy: 0.6661 - val_loss: 0.7997
Epoch 3/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - accuracy: 0.6766 - loss: 0.7934 - val_accuracy: 0.6718 - val_loss: 0.7780
Epoch 4/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 11s 50ms/step - accuracy: 0.6760 - loss: 0.7861 - val_accuracy: 0.6712 - val_loss: 0.7790
Epoch 5/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - accuracy: 0.6784 - loss: 0.7723 - val_accuracy: 0.6712 - val_loss: 0.7729
Epoch 6/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - accuracy: 0.6919 - loss: 0.7457 - val_accuracy: 0.6829 - val_loss: 0.7707
Epoch 7/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - accuracy: 0.6931 - loss: 0.7492 - val_accuracy: 0.6853 - val_loss: 0.7680
Epoch 8/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - accuracy: 0.6961 - loss: 0.7415 - 

# Load test data

In [18]:
# Load test data
test_data_path = "/kaggle/input/io-t-sleep-stage-classification-version-2/test_segment/test_segment"
test_folders = ["test001", "test002", "test003", "test004", "test005", "test006", "test007", "test008", "test009", "test010"]
test_csv_files = []

for folder in test_folders:
    folder_path = os.path.join(test_data_path, folder)
    test_csv_files.extend(glob.glob(f"{folder_path}/*.csv"))

# Preprocess data test

In [19]:
# Load and preprocess test data
df_test = pd.concat([pd.read_csv(file) for file in test_csv_files], ignore_index=True)
df_test[feature_cols] = scaler.transform(df_test[feature_cols])

# Reshape test data into sequence model

In [21]:
num_samples_test = len(df_test) // 480
X_test = df_test[feature_cols].values[: num_samples_test * 480].reshape(num_samples_test, 480, -1)

# Predict sleep states 

In [22]:
y_test_pred = np.argmax(model.predict(X_test), axis=1)
y_test_pred_labels = label_encoder.inverse_transform(y_test_pred)

220/220 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step


# Submission to Kaggle

In [23]:
# Load sample submission
submission_path = "/kaggle/input/io-t-sleep-stage-classification-version-2/sample_submission.csv"
df_submission = pd.read_csv(submission_path)

# Ensure y_test_pred_labels matches df_submission length
df_submission = df_submission.iloc[:len(y_test_pred_labels)].copy()
df_submission["labels"] = y_test_pred_labels

# Keep only id and labels columns
df_submission = df_submission[["id", "labels"]]

# Save final submission
final_submission_path = "/kaggle/working/submission.csv"
df_submission.to_csv(final_submission_path, index=False)
print("Submission file saved at:", final_submission_path)

Submission file saved at: /kaggle/working/submission.csv
